# Assignment 2: UNO Game AI
**Name:** Afraz Ahmad  
**Roll No:** i242598  
**GitHub:** https://github.com/AfrazAhmad11/UNO_GameAI

In [5]:
import random
import copy
import tkinter as tk
from tkinter import messagebox, simpledialog

In [6]:
class Card:
    """
    Represents a single UNO card.
    color: Red, Blue, Green, Yellow
    value: 0-9 or 'Skip'
    """
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        # Two cards are equal if same color and value
        return self.color == other.color and self.value == other.value

In [7]:
def generate_deck():
    """
    Generates a shuffled UNO deck.
    Contains: Red/Blue/Green/Yellow 0-9 + 2 Skip cards per color
    """
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    # Add number cards 0-9 for each color
    for color in colors:
        for number in range(10):
            deck.append(Card(color, number))

    # Add 2 Skip cards per color
    for color in colors:
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))

    # Shuffle the deck
    random.shuffle(deck)
    return deck

# Test deck
deck = generate_deck()
print(f"Total cards in deck: {len(deck)}")
print(f"First 5 cards: {deck[:5]}")

Total cards in deck: 48
First 5 cards: [Green 1, Green 4, Blue 6, Yellow 2, Blue Skip]


In [8]:
# Game State + Deal Card

In [9]:
def initialize_game():
    """
    Sets up the initial game state.
    Each player gets 5 cards. Top card is revealed.
    """
    deck = generate_deck()

    # Deal 5 cards to each player
    p1_hand = [deck.pop() for _ in range(5)]  # Player 1 - Minimax (Defensive)
    p2_hand = [deck.pop() for _ in range(5)]  # Player 2 - Expectimax (Offensive)
    p3_hand = [deck.pop() for _ in range(5)]  # Player 3 - User/AI

    # Pick a top card (must be a number card, not Skip)
    top_card = None
    while top_card is None:
        card = deck.pop()
        if card.value != 'Skip':
            top_card = card
        else:
            deck.insert(0, card)  # Put Skip back at bottom

    # Game state dictionary
    state = {
        'p1_hand': p1_hand,   # Player 1 hand
        'p2_hand': p2_hand,   # Player 2 hand
        'p3_hand': p3_hand,   # Player 3 hand
        'top_card': top_card, # Current top card on discard pile
        'deck': deck,         # Remaining draw deck
        'skip_next': None     # Tracks who is skipped
    }
    return state

# Test initialization
state = initialize_game()
print(f"Top Card: {state['top_card']}")
print(f"P1 Hand: {state['p1_hand']}")
print(f"P2 Hand: {state['p2_hand']}")
print(f"P3 Hand: {state['p3_hand']}")
print(f"Remaining deck: {len(state['deck'])} cards")

Top Card: Yellow 2
P1 Hand: [Blue Skip, Green 0, Blue 6, Green 6, Red Skip]
P2 Hand: [Green 1, Red 4, Red 3, Yellow 0, Green Skip]
P3 Hand: [Red Skip, Green 5, Green 2, Yellow 5, Blue 9]
Remaining deck: 32 cards


In [10]:
# Legal Move Generator

In [11]:
def get_valid_moves(hand, top_card):
    """
    Returns list of valid cards a player can play.
    Rule: Card must match top card's color OR number/value.
    """
    valid = []
    for card in hand:
        # Same color OR same number/value
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid

# Test
top = Card('Red', 5)
hand = [Card('Red', 3), Card('Blue', 5), Card('Green', 7), Card('Red', 'Skip'), Card('Yellow', 2)]
print(f"Top Card: {top}")
print(f"Valid moves: {get_valid_moves(hand, top)}")

Top Card: Red 5
Valid moves: [Red 3, Blue 5, Red Skip]


In [12]:
# State Transition /(apply move)

In [13]:
def apply_move(state, player, card):
    """
    Applies a move to the state and returns a new state.
    player: 'p1', 'p2', or 'p3'
    card: Card object or None (means draw)
    """
    # Deep copy so original state is not modified
    new_state = copy.deepcopy(state)
    hand_key = f'{player}_hand'

    if card is None:
        # DRAW: player draws 1 card from deck
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[hand_key].append(drawn)
    else:
        # PLAY: remove card from hand, set as top card
        new_state[hand_key] = [c for c in new_state[hand_key]
                                if not (c.color == card.color and c.value == card.value)]
        # Remove only one copy
        original = state[hand_key]
        new_hand = copy.deepcopy(original)
        for i, c in enumerate(new_hand):
            if c.color == card.color and c.value == card.value:
                new_hand.pop(i)
                break
        new_state[hand_key] = new_hand
        new_state['top_card'] = copy.deepcopy(card)

        # Handle Skip card
        if card.value == 'Skip':
            new_state['skip_next'] = True
        else:
            new_state['skip_next'] = False

    return new_state

print("apply_move function defined successfully.")

apply_move function defined successfully.


---
## Cell 7: Evaluation Function

### Formula:
**Score = 50 − 5(C_AI) + 2(C_opp) + 3(S)**

- **C_AI**: Number of cards in current player's hand (fewer = better)
- **C_opp**: Average cards held by opponents (more opponent cards = better for us)
- **S**: Number of Skip cards in hand (more skips = more control)

### Weight Tuning:
- **Defensive (P1)**: Higher penalty for own cards, higher reward for Skip (control)
- **Offensive (P2)**: Higher reward for opponent cards, focus on shedding own cards fast

In [14]:
def evaluate(state, player, strategy='defensive'):
    """
    Evaluation function for the given player.
    strategy: 'defensive' (P1 - Minimax) or 'offensive' (P2 - Expectimax)

    Base formula: Score = 50 - 5*C_AI + 2*C_opp + 3*S
    Weights are tuned per strategy.
    """
    hand_key = f'{player}_hand'
    players = ['p1', 'p2', 'p3']
    opponents = [p for p in players if p != player]

    # Cards in AI hand
    c_ai = len(state[hand_key])

    # Average cards in opponents' hands
    c_opp = sum(len(state[f'{p}_hand']) for p in opponents) / len(opponents)

    # Skip cards in hand
    s = sum(1 for card in state[hand_key] if card.value == 'Skip')

    if strategy == 'defensive':
        # Defensive: penalize own cards more, reward skips more
        # Focuses on not losing rather than winning fast
        w_ai   = 6   # Higher penalty for own cards
        w_opp  = 2   # Normal weight for opponents
        w_skip = 4   # Higher reward for skip (control)
        score = 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

    elif strategy == 'offensive':
        # Offensive: reward shedding cards fast, also reward hurting opponents
        # Focuses on winning quickly
        w_ai   = 5   # Normal penalty for own cards
        w_opp  = 3   # Higher reward for opponent cards (want them stuck)
        w_skip = 2   # Lower skip reward (less focused on control)
        score = 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

    else:
        # Default baseline formula
        score = 50 - 5 * c_ai + 2 * c_opp + 3 * s

    return round(score, 2)

# Test evaluation
state = initialize_game()
print(f"P1 Defensive Score: {evaluate(state, 'p1', 'defensive')}")
print(f"P2 Offensive Score: {evaluate(state, 'p2', 'offensive')}")

P1 Defensive Score: 30.0
P2 Offensive Score: 42.0
